# Load ENV Variable

In [126]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [125]:
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": text
    })

def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": text
    })

def chat(messages, system=None, stop_sequences=None, temperature=1.0):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    
    return message.content[0].text


In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_criterion": "Must include runtime, memory size, timeout, , and basic structure of
        AWS lambda configuration"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.


* Your entire response must be valid JSON and nothing else — no markdown fences, no commentary, no preamble. The first character of your response must be [ and the last character must be ].

"""
    messages = []
    add_user_message(messages, prompt)
    # add_assistant_message(messages, "Sure! Here's a dataset of tasks that require Python, JSON, or Regex for AWS-related tasks.")
    response = chat(messages)
    return json.loads(response)

In [111]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [118]:
def run_prompt(testcase):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:
{testcase["task"]}

* Respond only with Python, JSON, or a plain Regex
* And do not add any comments, commentary or explanation
* Your entire response must be valid JSON, Python or Regex and nothing else — no markdown fences, no commentary, no preamble, no ```json, no ```python, no ```regex, no ```code.
    Example of correct output for a Python task:
    def add(a, b):
        return a + b

    Example of INCORRECT output (do not do this):
    ```python
    def add(a, b):
        return a + b
    ```
"""
    messages = []
    add_user_message(messages, prompt)
    response = chat(messages)
    return response

In [119]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution Criterion:
<criterion>
{test_case["solution_criterion"]}
</criterion>

Output Format
Provide your evaluation as a structured JSON object with the following fields against the solution criterion, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Your entire response must be valid JSON and nothing else — no markdown fences, no commentary, no preamble.
Example response shape should be strictly like below:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages)
    return json.loads(eval_text)
    


In [120]:
# Functions to validate the output structure
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == 'json':
        return validate_json(response)
    elif format == 'python':
        return validate_python(response)
    else:   
        return validate_regex(response)



In [121]:
def run_test_case(testcase):
    """Calls run_prompt and then grades the result"""
    output = run_prompt(testcase)

    # grading
    model_grade = grade_by_model(testcase, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, testcase)
    score = (model_score + syntax_score)/2

    return {
        "output": output,
        "task": testcase["task"],
        "score": score,
        "reasoning": reasoning
    }

In [122]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each test case"""
    results = []
    
    for testcase in dataset:
        result = run_test_case(testcase)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average score: {average_score}")

    return results  



In [123]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 9.166666666666666


In [42]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# S3 Object Key Listing Function\n\n## Implementation\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\nfrom typing import Optional\n\n\ndef list_s3_objects(\n    bucket_name: str,\n    prefix: str = \"\",\n    region_name: Optional[str] = None,\n) -> list[str]:\n    \"\"\"\n    List all object keys in an S3 bucket matching the given prefix.\n\n    Args:\n        bucket_name: The name of the S3 bucket.\n        prefix:      The prefix string to filter objects (default: \"\" lists all).\n        region_name: Optional AWS region name (e.g., \"us-east-1\").\n\n    Returns:\n        A list of object key strings that match the given prefix.\n\n    Raises:\n        ValueError:          If bucket_name is empty.\n        NoCredentialsError:  If AWS credentials are not configured.\n        ClientError:         If the bucket doesn't exist or access is denied.\n    \"\"\"\n    if not bucket_name:\n        raise ValueError(\"bucket_na